# 3-Phase Frequent Itemset Extractor Training (Qwen 2.5-7B) — v3

**Pipeline:** SFT with Chain-of-Thought → DPO with Real LLM Failures *(GRPO skipped in v3)*

## What's different in v3 (LLM Council recommendations)
| Aspect | v2 | v3 (this notebook) |
|--------|----|--------------------|
| LoRA rank | r=64, alpha=16 (ratio 0.25) | r=**32**, alpha=**64** (ratio **2.0**) |
| LoRA dropout | 0 | **0.05** |
| Seq length | 4096 | **2048** (concise CoT fits) |
| SFT config | 2 epochs, lr=2e-4 | **3** epochs, lr=**1e-4**, eval every 50 steps |
| DPO config | 2 epochs | **1** epoch (overfits quickly) |
| GRPO | Full 200 steps | **SKIPPED** (add in v4 if needed) |
| Save method | `merged_4bit_forced` ❌ | **Adapter-only** via `save_pretrained` ✅ |
| Inference temp | 0.1 | **0.3** + top_k=50, top_p=0.90 |
| CoT format | Verbose with evidence rows | **Concise** column-grouped format |

## Phases
1. **SFT-CoT** — Teach the model to reason step-by-step using `<think>` tags, then output JSON
2. **DPO-Real** — Steer away from real mistakes LLMs actually made (hallucinated evidence rows, wrong counts)
3. **GRPO** *(skipped in v3)* — Can be added later as refinement pass

## Requirements
- GPU with ≥16 GB VRAM (T4, A100, L4, H200, etc.)
- Dataset on HuggingFace Hub (uploaded by `build_hf_dataset_v2.py`)
- ~1-2 hours total training time on H200/A100

In [ ]:
# ── Cell 1: Install dependencies (run ONCE, then restart kernel) ──────────────
# ⚠️ TLJH server has a working torch 2.7.1+cu118 at /opt/tljh/user/.
#    Installing unsloth pulls torch 2.10+ to ~/.local which SHADOWS system torch
#    and crashes (CUDA 11.8 vs CUDA 12.x mismatch).
#
#    Fix: install deps → delete ONLY core torch + nvidia cu12x → restart kernel.
#    We KEEP torchvision/torchaudio (unsloth imports them, not used for training).
#    The cell is IDEMPOTENT: safe to re-run, won't re-pull torch if deps exist.

import os, glob, shutil, subprocess

USER_SP = os.path.expanduser("~/.local/lib/python3.12/site-packages")

# ── Step 1: Only install if unsloth is NOT already present ────────────────────
unsloth_dir = os.path.join(USER_SP, "unsloth")
if not os.path.isdir(unsloth_dir):
    print("📦 First run — installing ML packages...")
    subprocess.check_call(
        "pip install unsloth trl datasets transformers accelerate "
        "bitsandbytes huggingface_hub peft safetensors sentencepiece protobuf -q".split()
    )
    print("📦 Install complete.")
else:
    print("✅ Packages already installed — skipping pip install")

# ── Step 2: Remove ONLY core torch + nvidia (keep torchvision/torchaudio) ────
# We delete: torch, torch-*, nvidia* (cu12x CUDA libs that conflict with cu118)
# We keep:   torchvision, torchaudio, triton (unsloth imports them, harmless)
removed = []
for pattern in ["torch", "torch-*"]:
    for p in glob.glob(os.path.join(USER_SP, pattern)):
        basename = os.path.basename(p)
        # Skip torchvision and torchaudio — unsloth needs them importable
        if basename.startswith("torchvision") or basename.startswith("torchaudio"):
            continue
        shutil.rmtree(p, ignore_errors=True)
        removed.append(basename)

# Remove nvidia cu12x runtime libs (conflict with system CUDA 11.8)
for p in glob.glob(os.path.join(USER_SP, "nvidia*")):
    shutil.rmtree(p, ignore_errors=True)
    removed.append(os.path.basename(p))

if removed:
    print(f"🗑️  Cleaned from ~/.local: {removed}")
else:
    print("✅ No user-level torch/nvidia to clean")

# ── Step 3: Verify system torch is reachable ─────────────────────────────────
assert not os.path.exists(os.path.join(USER_SP, "torch")), \
    f"❌ FAILED: torch still in {USER_SP}/torch — please delete manually"

r = subprocess.run(
    ["python3", "-c", "import torch; print(torch.__version__, torch.__file__)"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"✅ System torch: {r.stdout.strip()}")
else:
    print(f"⚠️  System torch not found: {r.stderr.strip()[:200]}")

print("\n" + "=" * 60)
print("⚠️  RESTART THE KERNEL (Kernel → Restart)")
print("   Then run cell 2 (CONFIG) onwards. Re-running this cell is safe.")
print("=" * 60)

In [ ]:
# ── Cell 2: CONFIG — edit this cell only ─────────────────────────────────────
# v3 (2026-03-09): Updated per LLM Council unanimous recommendations
# Key changes: r=32, alpha=64, dropout=0.05, lr=1e-4, seq_len=2048,
#              3 SFT epochs, 1 DPO epoch, GRPO skipped, adapter-only save
CONFIG = {
    # ── Model ────────────────────────────────────────────────────────────────
    "base_model":        "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "max_seq_length":    2048,    # v3: was 4096, reduced to match concise CoT format
    "load_in_4bit":      True,

    # ── Dataset (HuggingFace Hub) ─────────────────────────────────────────────
    "hf_dataset":        "OliverSlivka/itemset-extraction-v2",
    "hf_token":          "",    # paste HF token here, or set env HF_TOKEN

    # ── LoRA (v3: alpha/r ratio = 2.0 for stronger updates) ──────────────────
    "lora_r":            32,     # v3: was 64 → 32 (smaller rank, avoids overfitting)
    "lora_alpha":        64,     # v3: was 16 → 64 (alpha/r = 2.0 for stronger signal)
    "lora_dropout":      0.05,   # v3: was 0 → 0.05 (regularization)
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"],

    # ── Phase 1: SFT with CoT ────────────────────────────────────────────────
    "sft_epochs":        3,      # v3: was 2 → 3 (with lower LR, more epochs needed)
    "sft_lr":            1e-4,   # v3: was 2e-4 → 1e-4 (gentler updates)
    "sft_batch_size":    2,      # reduce to 1 if OOM
    "sft_grad_accum":    4,      # effective batch = sft_batch_size × sft_grad_accum
    "sft_warmup_ratio":  0.10,   # v3: was 0.05 → 0.10
    "sft_weight_decay":  0.01,   # v3: added weight decay
    "sft_output_dir":    "./sft_checkpoint",

    # ── Phase 2: DPO with Real Failures ──────────────────────────────────────
    "dpo_epochs":        1,      # v3: was 2 → 1 (DPO overfits quickly)
    "dpo_lr":            5e-5,
    "dpo_beta":          0.1,
    "dpo_batch_size":    1,
    "dpo_grad_accum":    4,
    "dpo_output_dir":    "./dpo_checkpoint",

    # ── Phase 3: GRPO — SKIPPED in v3 (council: skip until SFT+DPO baseline works)
    # "grpo_max_steps":    200,
    # "grpo_lr":           5e-6,
    # "grpo_batch_size":   1,
    # "grpo_grad_accum":   4,
    # "grpo_num_generations": 4,
    # "grpo_max_completion_length": 2048,
    # "grpo_output_dir":   "./grpo_checkpoint",

    # ── Output / Push ─────────────────────────────────────────────────────────
    "hf_model_repo":     "OliverSlivka/qwen2.5-7b-itemset-extractor",
    "push_to_hub":       True,
}

print("✅ CONFIG loaded (v3 — council recommendations)")
print(f"   Model: {CONFIG['base_model']}")
print(f"   Seq length: {CONFIG['max_seq_length']}")
print(f"   LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, dropout={CONFIG['lora_dropout']}")
print(f"   SFT: {CONFIG['sft_epochs']} epochs @ lr={CONFIG['sft_lr']}")
print(f"   DPO: {CONFIG['dpo_epochs']} epoch @ lr={CONFIG['dpo_lr']}, beta={CONFIG['dpo_beta']}")
print(f"   GRPO: SKIPPED (v3)")
print(f"   Dataset: {CONFIG['hf_dataset']}")

In [ ]:
# ── Cell 3: GPU check + imports ───────────────────────────────────────────────
# Disable TF/JAX BEFORE any transformers/datasets import.
# The TLJH system has TensorFlow + Keras 3 installed; transformers detects TF
# as available and tries to import TFPreTrainedModel → Keras 3 crash.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # suppress TF oneDNN spam

import gc, json, re, torch
from datasets import load_dataset
from huggingface_hub import login

# HF login
hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace logged in")
else:
    print("⚠️  No HF token — will prompt for login if needed")
    try:
        login()
    except Exception:
        print("   Skipping login — set hf_token in CONFIG or run `huggingface-cli login`")

# GPU info
if torch.cuda.is_available():
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    raise RuntimeError("❌ No GPU found — connect a GPU runtime before continuing.")

In [ ]:
# ── Cell 4: Load datasets from HuggingFace ────────────────────────────────────
# The v2 dataset has 3 configs: sft, dpo, grpo
print("📂 Loading datasets from HuggingFace Hub...")

sft_dataset  = load_dataset(CONFIG["hf_dataset"], "sft")
dpo_dataset  = load_dataset(CONFIG["hf_dataset"], "dpo")
grpo_dataset = load_dataset(CONFIG["hf_dataset"], "grpo")

print(f"✅ SFT:  {len(sft_dataset['train']):>4d} train / {len(sft_dataset['validation']):>3d} val")
print(f"✅ DPO:  {len(dpo_dataset['train']):>4d} train / {len(dpo_dataset['validation']):>3d} val")
print(f"✅ GRPO: {len(grpo_dataset['train']):>4d} train / {len(grpo_dataset['validation']):>3d} val")

In [ ]:
# ── Cell 5: Quick data preview ────────────────────────────────────────────────
example = sft_dataset["train"][0]
print("═" * 60)
print("SFT Example (messages format):")
print("═" * 60)
for msg in example["messages"]:
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 300:
        content = content[:300] + f"... ({len(content)} chars total)"
    print(f"\n[{role}]")
    print(content)

print("\n" + "═" * 60)
print("DPO Example (prompt/chosen/rejected):")
print("═" * 60)
dpo_ex = dpo_dataset["train"][0]
print(f"Prompt: {len(dpo_ex['prompt'])} messages")
print(f"Chosen: {len(dpo_ex['chosen'][0]['content'])} chars")
print(f"Rejected: {len(dpo_ex['rejected'][0]['content'])} chars")

---
## Phase 1 — SFT with Chain-of-Thought 🎓

Teaches the model to **reason step-by-step** using `<think>` tags with **v3 concise format**:
```
<think>
Dataset: 7 rows × 12 cols, min_support=3
## SCAN 1: Singles by column
age: 15=3(R1,R2,R7)✓ | 16=2✗ | 17=4(R3,R4,R5,R6)✓
medu: 4=5(R1,R3,R5,R6,R7)✓ | 1=2✗
## SCAN 2: Pairs
{age:15,medu:4}: R1,R7 → 2✗
{age:17,medu:4}: R3,R5,R6 → 3✓
## RESULT SUMMARY: 5 singles + 3 pairs + 1 triple = 9 itemsets
</think>
[{"itemset": [...], "count": N, "rows": ["Row 1", ...]}]
```

**v3 changes from council:**
- Concise CoT format (column-grouped, no evidence_rows in think)
- 3 epochs at lr=1e-4 (was 2 epochs at 2e-4)
- LoRA r=32, alpha=64 (ratio 2.0), dropout=0.05
- `load_best_model_at_end=True` with eval every 50 steps
- Uses `train_on_responses_only` — only trains on assistant content

In [ ]:
# ── Cell 7: Load model + LoRA ─────────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["base_model"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = CONFIG["load_in_4bit"],
    dtype          = None,  # auto: bfloat16 on Ampere+, float16 on older
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = CONFIG["lora_r"],
    lora_alpha                = CONFIG["lora_alpha"],
    target_modules            = CONFIG["lora_target_modules"],
    lora_dropout              = CONFIG["lora_dropout"],  # v3: 0.05 (was 0)
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",
    random_state              = 42,
)

model.print_trainable_parameters()

In [ ]:
# ── Cell 8: SFT training ─────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Pre-format: apply chat template to messages → plain text column.
# train_on_responses_only needs a text dataset (not raw messages dict),
# otherwise _tokenize_fn inside unsloth_zoo gets an empty list → IndexError.
def apply_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples["messages"]
        ]
    }

sft_train_fmt = sft_dataset["train"].map(
    apply_template, batched=True,
    remove_columns=sft_dataset["train"].column_names
)
sft_val_fmt = sft_dataset["validation"].map(
    apply_template, batched=True,
    remove_columns=sft_dataset["validation"].column_names
)
print(f"✅ Dataset formatted — train: {len(sft_train_fmt)}, val: {len(sft_val_fmt)}")
print(f"   Sample (first 200 chars): {sft_train_fmt[0]['text'][:200]}")

sft_trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = sft_train_fmt,
    eval_dataset    = sft_val_fmt,
    args            = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = CONFIG["max_seq_length"],
        num_train_epochs            = CONFIG["sft_epochs"],
        per_device_train_batch_size = CONFIG["sft_batch_size"],
        gradient_accumulation_steps = CONFIG["sft_grad_accum"],
        learning_rate               = CONFIG["sft_lr"],
        lr_scheduler_type           = "cosine",
        warmup_ratio                = CONFIG["sft_warmup_ratio"],  # v3: 0.10 (was 0.05)
        weight_decay                = CONFIG["sft_weight_decay"],  # v3: 0.01 (new)
        optim                       = "adamw_8bit",
        bf16                        = True,
        fp16                        = False,
        logging_steps               = 10,
        eval_strategy               = "steps",       # v3: per-step eval for early stopping
        eval_steps                  = 50,             # v3: eval every 50 steps
        save_strategy               = "steps",
        save_steps                  = 50,
        save_total_limit            = 3,
        load_best_model_at_end      = True,           # v3: keep best checkpoint
        metric_for_best_model       = "eval_loss",
        output_dir                  = CONFIG["sft_output_dir"],
        report_to                   = "none",
        seed                        = 42,
    ),
)

# Only train on assistant responses — mask system + user tokens
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("🎓 Starting SFT training with Chain-of-Thought (v3 config)...")
print(f"   Epochs: {CONFIG['sft_epochs']}, LR: {CONFIG['sft_lr']}, Warmup: {CONFIG['sft_warmup_ratio']}")
sft_result = sft_trainer.train()
print(f"✅ SFT done! Final loss: {sft_result.training_loss:.4f}")

In [ ]:
# ── Cell 9: Save SFT checkpoint (ADAPTER ONLY — v3 fix) ─────────────────────
# v3 CRITICAL FIX: Save adapter weights only, NOT merged_4bit_forced.
# Council finding: merged_4bit_forced destroys LoRA adapter structure,
# making it impossible to stack fresh LoRA for DPO phase.
# Adapter save is also 10x smaller (~65MB vs ~5GB).
model.save_pretrained(CONFIG["sft_output_dir"])
tokenizer.save_pretrained(CONFIG["sft_output_dir"])
print(f"💾 SFT adapter saved → {CONFIG['sft_output_dir']}")

# Evaluate SFT before proceeding to DPO (v3: eval gate)
print("\n⚠️  EVALUATION GATE: Run a quick eval on the SFT checkpoint before DPO.")
print("   If SFT quality is poor (F1 < 0.4), re-check training data quality.")

# Free memory before DPO
del model, sft_trainer
gc.collect()
torch.cuda.empty_cache()
print(f"🧹 Memory freed. VRAM available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

---
## Phase 2 — DPO with Real LLM Failures 🎯

Uses **actual mistakes** from GPT-4.1-mini, GPT-4.1-nano, o4-mini, GPT-4o:
- **Chosen** = Apriori ground truth with CoT reasoning
- **Rejected** = Real LLM outputs that failed validation (99.5% error = hallucinated evidence rows)

Why real failures beat synthetic:
- Match the **actual error distribution** LLMs produce
- Model learns to avoid the exact mistakes it would naturally make
- 606+ real DPO pairs from 313 unique datasets

**v3 changes from council:**
- 1 epoch only (was 2) — DPO overfits quickly with real failures
- `warmup_ratio=0.10` (was 0.05)
- Reference model = base + SFT adapter merged (adapter-only save preserves this)
- Fresh LoRA applied on top for DPO fine-tuning

In [ ]:
# ── Cell 11: Reload base model + merge SFT adapter + fresh LoRA for DPO ──────
# v3: Load base model, apply SFT adapter, then add fresh LoRA for DPO.
# This preserves the SFT knowledge while allowing DPO to learn preferences.
from peft import PeftModel

# Load base model fresh
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["base_model"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = CONFIG["load_in_4bit"],
    dtype          = None,
)

# Merge the SFT adapter into the base
model = PeftModel.from_pretrained(model, CONFIG["sft_output_dir"])
model = model.merge_and_unload()
print("✅ SFT adapter merged into base model")

# Apply fresh LoRA for DPO training
model = FastLanguageModel.get_peft_model(
    model,
    r                         = CONFIG["lora_r"],
    lora_alpha                = CONFIG["lora_alpha"],
    target_modules            = CONFIG["lora_target_modules"],
    lora_dropout              = CONFIG["lora_dropout"],  # v3: 0.05
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",
    random_state              = 42,
)

model.print_trainable_parameters()
print("✅ Fresh LoRA applied — ready for DPO")

In [ ]:
# ── Cell 12: DPO training with real failures ─────────────────────────────────
from trl import DPOTrainer, DPOConfig

# Format DPO data: apply chat template to prompt, extract content from chosen/rejected
def format_dpo(examples):
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        prompts.append(
            tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        )
        chosens.append(chosen[0]["content"])
        rejecteds.append(rejected[0]["content"])
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dpo_train = dpo_dataset["train"].map(
    format_dpo, batched=True, remove_columns=dpo_dataset["train"].column_names
)
dpo_val = dpo_dataset["validation"].map(
    format_dpo, batched=True, remove_columns=dpo_dataset["validation"].column_names
)

dpo_trainer = DPOTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dpo_train,
    eval_dataset     = dpo_val,
    args             = DPOConfig(
        beta                        = CONFIG["dpo_beta"],
        max_length                  = CONFIG["max_seq_length"],
        max_prompt_length           = CONFIG["max_seq_length"] // 2,
        num_train_epochs            = CONFIG["dpo_epochs"],   # v3: 1 epoch (was 2)
        per_device_train_batch_size = CONFIG["dpo_batch_size"],
        gradient_accumulation_steps = CONFIG["dpo_grad_accum"],
        learning_rate               = CONFIG["dpo_lr"],
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.10,   # v3: was 0.05
        optim                       = "adamw_8bit",
        bf16                        = True,
        fp16                        = False,
        gradient_checkpointing      = False,  # Unsloth handles this
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 2,
        output_dir                  = CONFIG["dpo_output_dir"],
        report_to                   = "none",
        seed                        = 42,
    ),
)

print("🎯 Starting DPO training with real LLM failures (v3: 1 epoch)...")
dpo_result = dpo_trainer.train()
print(f"✅ DPO done! Final loss: {dpo_result.training_loss:.4f}")

In [ ]:
# ── Cell 13: Save DPO checkpoint (ADAPTER ONLY — v3 fix) ────────────────────
# v3: Adapter-only save. NEVER use merged_4bit_forced.
model.save_pretrained(CONFIG["dpo_output_dir"])
tokenizer.save_pretrained(CONFIG["dpo_output_dir"])
print(f"💾 DPO adapter saved → {CONFIG['dpo_output_dir']}")

# v3: GRPO is SKIPPED — jump directly to Cell 19 (inference test)
# If you want to try GRPO in future, uncomment the GRPO cells below.
print("\n📌 v3: GRPO skipped (council recommendation). Proceeding to inference test.")
print("   Next: Run Cell 19 (inference test) → Cell 20 (push to hub)")

---
## Phase 3 — GRPO with Apriori Reward 🔬 *(SKIPPED in v3)*

**⚠️ v3 Council Decision: SKIP GRPO** until SFT+DPO baseline achieves F1 ≥ 0.60.

Rationale (unanimous from 4 frontier models):
- GRPO adds complexity and instability without proven benefit for this task
- SFT+DPO should be sufficient to reach 80% F1 target
- GRPO can be added in v4 if needed as a refinement pass

**If you want to try GRPO anyway**, uncomment the cells below. But run eval on DPO first!

In [ ]:
# ── Cell 15: GRPO reward functions ───────────────────────────────────────────

def _extract_json(text):
    """Extract JSON array from model response, handling <think> tags."""
    # Strip <think> block if present
    if "</think>" in text:
        text = text.split("</think>", 1)[-1].strip()
    # Find JSON array
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            return None
    return None


def json_format_reward(completions, **kwargs):
    """Reward for valid JSON with correct schema: itemset, count, rows."""
    rewards = []
    for text in completions:
        parsed = _extract_json(text)
        if parsed is None:
            rewards.append(0.0)
        elif not isinstance(parsed, list) or len(parsed) == 0:
            rewards.append(0.2)
        elif all(
            isinstance(x, dict) and "itemset" in x and "count" in x and "rows" in x
            for x in parsed
        ):
            rewards.append(1.0)
        elif all(isinstance(x, dict) and "itemset" in x for x in parsed):
            rewards.append(0.5)
        else:
            rewards.append(0.2)
    return rewards


def itemset_f1_reward(completions, ground_truth, **kwargs):
    """F1 score of predicted itemsets vs Apriori ground truth."""
    rewards = []
    for text, gt_str in zip(completions, ground_truth):
        predicted = _extract_json(text)
        try:
            gt = json.loads(gt_str)
        except (json.JSONDecodeError, TypeError):
            rewards.append(0.0)
            continue

        if predicted is None:
            rewards.append(0.0)
            continue

        pred_sets = {frozenset(x["itemset"]) for x in predicted if isinstance(x, dict) and "itemset" in x}
        true_sets = {frozenset(x["itemset"]) for x in gt if isinstance(x, dict) and "itemset" in x}

        if not true_sets:
            rewards.append(1.0 if not pred_sets else 0.0)
            continue

        tp = len(pred_sets & true_sets)
        precision = tp / len(pred_sets) if pred_sets else 0.0
        recall = tp / len(true_sets)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        rewards.append(f1)
    return rewards


def count_accuracy_reward(completions, ground_truth, **kwargs):
    """Fraction of matched itemsets with correct count values."""
    rewards = []
    for text, gt_str in zip(completions, ground_truth):
        predicted = _extract_json(text)
        try:
            gt = json.loads(gt_str)
        except (json.JSONDecodeError, TypeError):
            rewards.append(0.0)
            continue

        if predicted is None:
            rewards.append(0.0)
            continue

        gt_map = {}
        for x in gt:
            if isinstance(x, dict) and "itemset" in x:
                gt_map[frozenset(x["itemset"])] = x.get("count", 0)

        correct = total = 0
        for p in predicted:
            if not isinstance(p, dict):
                continue
            key = frozenset(p.get("itemset", []))
            if key in gt_map:
                total += 1
                if p.get("count") == gt_map[key]:
                    correct += 1

        rewards.append(correct / total if total > 0 else 0.0)
    return rewards


def thinking_reward(completions, **kwargs):
    """Reward for structured reasoning in <think> tags."""
    rewards = []
    for text in completions:
        if "<think>" in text and "</think>" in text:
            think = text.split("<think>", 1)[1].split("</think>", 1)[0]
            # Reward scales with thinking quality
            score = min(1.0, len(think) / 300)
            # Bonus for structured analysis markers
            if any(marker in think.lower() for marker in ["→", "count", "singles", "pairs", "✓"]):
                score = min(1.0, score + 0.2)
            rewards.append(score)
        else:
            rewards.append(0.0)
    return rewards


print("✅ GRPO reward functions defined:")
print("   1. json_format_reward     — Valid JSON with itemset/count/rows schema")
print("   2. itemset_f1_reward      — F1 vs Apriori ground truth")
print("   3. count_accuracy_reward  — Correct counts for matched itemsets")
print("   4. thinking_reward        — Structured <think> reasoning")

In [ ]:
# ── Cell 16: Load DPO checkpoint + GRPO training ─────────────────────────────
from trl import GRPOTrainer, GRPOConfig

# Reload from DPO checkpoint
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["dpo_output_dir"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = CONFIG["load_in_4bit"],
    dtype          = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = CONFIG["lora_r"],
    lora_alpha                = CONFIG["lora_alpha"],
    target_modules            = CONFIG["lora_target_modules"],
    lora_dropout              = 0,
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",
    random_state              = 42,
)

# Format GRPO dataset: apply chat template to prompt
def format_grpo(examples):
    prompts = []
    for prompt_msgs in examples["prompt"]:
        prompts.append(
            tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        )
    return {"prompt": prompts, "ground_truth": examples["ground_truth"]}

grpo_train = grpo_dataset["train"].map(
    format_grpo, batched=True, remove_columns=grpo_dataset["train"].column_names
)

grpo_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        json_format_reward,
        itemset_f1_reward,
        count_accuracy_reward,
        thinking_reward,
    ],
    args             = GRPOConfig(
        max_steps                   = CONFIG["grpo_max_steps"],
        per_device_train_batch_size = CONFIG["grpo_batch_size"],
        gradient_accumulation_steps = CONFIG["grpo_grad_accum"],
        learning_rate               = CONFIG["grpo_lr"],
        num_generations             = CONFIG["grpo_num_generations"],
        max_completion_length       = CONFIG["grpo_max_completion_length"],
        warmup_ratio                = 0.05,
        optim                       = "adamw_8bit",
        bf16                        = True,
        fp16                        = False,
        logging_steps               = 5,
        save_steps                  = 50,
        output_dir                  = CONFIG["grpo_output_dir"],
        report_to                   = "none",
        seed                        = 42,
    ),
    train_dataset    = grpo_train,
)

print(f"🔬 Starting GRPO training for {CONFIG['grpo_max_steps']} steps...")
print(f"   Reward functions: json_format, itemset_f1, count_accuracy, thinking")
print(f"   Generations per prompt: {CONFIG['grpo_num_generations']}")
grpo_result = grpo_trainer.train()
print(f"✅ GRPO done!")

In [ ]:
# ── Cell 17: Save GRPO model ─────────────────────────────────────────────────
model.save_pretrained_merged(
    CONFIG["grpo_output_dir"] + "/final",
    tokenizer,
    save_method = "merged_4bit_forced",
)
print(f"💾 GRPO final model saved → {CONFIG['grpo_output_dir']}/final")

---
## Inference Test 🧪

Quick sanity check on a sample CSV to verify the model produces valid JSON with reasoning.

In [ ]:
# ── Cell 19: Quick inference test (v3: temp=0.3, top_k=50) ───────────────────
# v3: Reload DPO adapter on base model for inference
from peft import PeftModel

# Load base + merge SFT adapter + merge DPO adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CONFIG["base_model"],
    max_seq_length = CONFIG["max_seq_length"],
    load_in_4bit   = True,
    dtype          = None,
)

# Stack adapters: base → SFT → DPO
model = PeftModel.from_pretrained(model, CONFIG["sft_output_dir"])
model = model.merge_and_unload()
model = PeftModel.from_pretrained(model, CONFIG["dpo_output_dir"])
model = model.merge_and_unload()

FastLanguageModel.for_inference(model)
print("✅ Model loaded with SFT + DPO adapters merged")

SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: \"Row N\" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{\"itemset\": [\"item1\", \"item2\"], \"count\": N, \"rows\": [\"Row 1\", \"Row 3\"]}]'
)

SAMPLE_CSV = """Row 1: bread, milk, eggs
Row 2: bread, butter, jam
Row 3: milk, eggs, cheese
Row 4: bread, milk, eggs, butter
Row 5: bread, eggs"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Find all frequent itemsets with minimum support count = 2 in this dataset:\n\n{SAMPLE_CSV}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

# v3 inference settings (council recommendation):
#   temp=0.3 (escape attractor loops), top_k=50 (prune long tail), top_p=0.90
outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 1024,
    temperature    = 0.3,     # v3: was 0.1
    top_k          = 50,      # v3: added
    top_p          = 0.90,    # v3: added
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("─── Model Output ───")
print(response)

# Validate
try:
    json_text = response
    if "</think>" in json_text:
        json_text = json_text.split("</think>", 1)[-1].strip()
    parsed = json.loads(re.search(r'\[.*\]', json_text, re.DOTALL).group())
    print(f"\n✅ Valid JSON — {len(parsed)} itemsets found")
    for item in parsed[:5]:
        print(f"  {item.get('itemset')}  count={item.get('count')}  rows={item.get('rows')}")
    if len(parsed) > 5:
        print(f"  ... and {len(parsed) - 5} more")
except Exception as e:
    print(f"\n⚠️  JSON parse failed: {e}")

In [ ]:
# ── Cell 20: Push final model to HuggingFace Hub ─────────────────────────────
# v3: Push as LoRA adapter (save_method="lora"), NOT merged_4bit_forced.
# Council finding: merged_4bit_forced destroys adapter structure and
# produces irreproducible weights. Adapter push is faster and smaller.
import os

if CONFIG["push_to_hub"]:
    hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")

    print(f"🚀 Pushing DPO model to HF Hub: {CONFIG['hf_model_repo']}")

    # Push adapter weights (small, ~65MB)
    model.push_to_hub(
        CONFIG["hf_model_repo"],
        token = hf_token,
    )
    tokenizer.push_to_hub(
        CONFIG["hf_model_repo"],
        token = hf_token,
    )

    print(f"✅ Adapter pushed to: https://huggingface.co/{CONFIG['hf_model_repo']}")
    print(f"   To load: model = PeftModel.from_pretrained(base_model, '{CONFIG['hf_model_repo']}')")

    # Optional: also push merged 16-bit model (for easier loading, ~14GB)
    # Uncomment if you want a standalone model that doesn't need PeftModel:
    # model.push_to_hub_merged(
    #     CONFIG["hf_model_repo"] + "-merged",
    #     tokenizer,
    #     save_method = "merged_16bit",
    #     token = hf_token,
    # )
    # print(f"   Merged 16-bit also pushed to: {CONFIG['hf_model_repo']}-merged")
else:
    print("ℹ️  push_to_hub=False — model only saved locally")
    print(f"   SFT adapter: {CONFIG['sft_output_dir']}")
    print(f"   DPO adapter: {CONFIG['dpo_output_dir']}")